# Eval-Driven Prompt Refinement

This notebook demonstrates a self-contained workflow that refines prompts through an evaluate-and-iterate loop.
It combines four key workflow patterns in a single cohesive scenario:

- **Looping with conditional exit** — retry until the evaluation passes or max iterations are reached
- **Branching** — approve, retry with feedback, or escalate to a human
- **State management** — track prompt history, evaluation scores, and iteration count
- **Human-in-the-loop** — escalation path when automated refinement fails

The workflow uses a mock evaluator so **no API keys are needed**.

### How it works

```
StartEvent
    │
    ▼
generate  ◄──── GeneratePromptEvent (automated retry)
    │          ◄──── HumanFeedbackResponse (HITL resume)
    │
    ▼
EvalResultEvent
    │
    ▼
evaluate_and_decide
    ├── score >= threshold  ──►  handle_approval  ──►  StopEvent
    ├── retries remaining   ──►  GeneratePromptEvent (loop)
    └── retries exhausted   ──►  HumanFeedbackRequired (HITL)
                                        │
                                        ▼
                               HumanFeedbackResponse → generate
```

In [1]:
!uv pip install llama-index-workflows -q

## 1. Define Events

Each event type defines a distinct edge in the workflow graph.

In [2]:
from workflows.events import (
    Event,
    HumanResponseEvent,
    InputRequiredEvent,
    StartEvent,
    StopEvent,
)


class GeneratePromptEvent(Event):
    """Triggers prompt generation. Carries optional feedback from a previous evaluation."""

    feedback: str = ""


class EvalResultEvent(Event):
    """Carries the evaluation outcome for a candidate prompt."""

    prompt: str
    score: float
    feedback: str


class ApprovedEvent(Event):
    """The prompt passed evaluation."""

    prompt: str
    score: float


class HumanFeedbackRequired(InputRequiredEvent):
    """Subclass of InputRequiredEvent so the caller can distinguish this prompt."""

    prompt: str
    history: list[dict]


class HumanFeedbackResponse(HumanResponseEvent):
    """Carries the human's feedback to resume the refinement loop."""

    response: str

## 2. Mock Evaluator

A deterministic evaluator that scores prompts based on simple heuristics.
Each criterion awards partial credit so the score increases as the prompt improves.

In [3]:
DATASET = [
    {"input": "What is 2+2?", "expected": "4"},
    {"input": "Summarize: The cat sat on the mat.", "expected": "A cat sat on a mat."},
    {"input": "Translate to French: Hello", "expected": "Bonjour"},
]

CRITERIA = [
    ("specific", ["specific", "precise", "exact", "concise"]),
    ("structured", ["step", "first", "then", "finally", "1.", "2."]),
    ("role", ["you are", "act as", "your role", "as a"]),
    ("examples", ["example", "e.g.", "for instance", "such as"]),
    ("constraints", ["must", "do not", "avoid", "ensure", "always"]),
]


class MockEvaluator:
    """Scores prompts 0.0-1.0 based on keyword heuristics."""

    def __init__(self, dataset: list[dict]):
        self.dataset = dataset

    def evaluate(self, prompt: str) -> tuple[float, str]:
        lower = prompt.lower()
        hits = []
        misses = []
        for name, keywords in CRITERIA:
            if any(kw in lower for kw in keywords):
                hits.append(name)
            else:
                misses.append(name)

        score = len(hits) / len(CRITERIA)

        if misses:
            feedback = f"Missing qualities: {', '.join(misses)}. Try adding language that is {', '.join(misses)}."
        else:
            feedback = "All criteria met."

        return round(score, 2), feedback

## 3. Mock Prompt Generator

A template-based generator that incorporates feedback to produce
incrementally better prompts. It adds at most **two** improvements per
iteration, simulating realistic gradual refinement.

In [4]:
BASE_PROMPT = "Answer the question."

REFINEMENT_ADDITIONS = {
    "specific": "Be precise and concise in your answer.",
    "structured": "First, understand the question. Then, provide a step-by-step answer.",
    "role": "You are a helpful and knowledgeable assistant.",
    "examples": "For instance, if asked about math, show your working (e.g. 2+2=4).",
    "constraints": "You must answer accurately. Do not guess. Always verify your reasoning.",
}

MAX_ADDITIONS_PER_ITERATION = 2


def generate_prompt(feedback: str, previous_prompt: str | None = None) -> str:
    """Build a prompt by appending up to MAX_ADDITIONS_PER_ITERATION sentences
    that address missing qualities mentioned in the feedback."""
    base = previous_prompt or BASE_PROMPT
    additions = []
    lower_feedback = feedback.lower()
    for quality, sentence in REFINEMENT_ADDITIONS.items():
        if quality in lower_feedback and sentence not in base:
            additions.append(sentence)
            if len(additions) >= MAX_ADDITIONS_PER_ITERATION:
                break
    if additions:
        return base + " " + " ".join(additions)
    return base

## 4. Define the Workflow

The `PromptRefinementWorkflow` ties everything together. HITL is modeled by `HumanFeedbackRequired` (a subclass of `InputRequiredEvent`) from the decide step and `HumanFeedbackResponse` (a `HumanResponseEvent`) feeding back into `generate`, without extra routing steps.

| Step | Consumes | Produces | Pattern |
|------|----------|----------|---------|
| `generate` | `StartEvent`, `GeneratePromptEvent`, `HumanFeedbackResponse` | `EvalResultEvent` | **Loop** (re-entry); **HITL** resume |
| `evaluate_and_decide` | `EvalResultEvent` | `ApprovedEvent` / `GeneratePromptEvent` / `HumanFeedbackRequired` | **Branching** (3-way) |
| `handle_approval` | `ApprovedEvent` | `StopEvent` | Terminal |

In [5]:
from pydantic import BaseModel
from workflows import Context, Workflow, step


class RefinementState(BaseModel):
    iteration: int = 0
    history: list[dict] = []
    current_prompt: str = ""


class PromptRefinementWorkflow(Workflow):
    def __init__(
        self,
        evaluator: MockEvaluator,
        max_retries: int = 3,
        threshold: float = 0.8,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.evaluator = evaluator
        self.max_retries = max_retries
        self.threshold = threshold

    # -- Step 1: Generate a candidate prompt --------------------------------

    @step
    async def generate(
        self,
        ctx: Context[RefinementState],
        ev: StartEvent | GeneratePromptEvent | HumanFeedbackResponse,
    ) -> EvalResultEvent:
        if isinstance(ev, HumanFeedbackResponse):
            async with ctx.store.edit_state() as state:
                state.iteration = 0
            feedback = ev.response
            print(f"  Human feedback received: {ev.response!r}")
        elif isinstance(ev, GeneratePromptEvent):
            feedback = ev.feedback
        else:
            feedback = ""

        async with ctx.store.edit_state() as state:
            state.iteration += 1
            previous = state.current_prompt or None
            candidate = generate_prompt(feedback, previous)
            state.current_prompt = candidate

        print(f"  [iteration {state.iteration}] Generated prompt: {candidate!r}")

        score, eval_feedback = self.evaluator.evaluate(candidate)
        return EvalResultEvent(prompt=candidate, score=score, feedback=eval_feedback)

    # -- Step 2: Decide — approve, retry, or HITL ---------------------------

    @step
    async def evaluate_and_decide(
        self, ctx: Context[RefinementState], ev: EvalResultEvent
    ) -> ApprovedEvent | GeneratePromptEvent | HumanFeedbackRequired:
        async with ctx.store.edit_state() as state:
            state.history.append(
                {"prompt": ev.prompt, "score": ev.score, "feedback": ev.feedback}
            )
            iteration = state.iteration
            history = list(state.history)

        print(
            f"  [iteration {iteration}] Score: {ev.score} (threshold: {self.threshold})"
        )

        if ev.score >= self.threshold:
            return ApprovedEvent(prompt=ev.prompt, score=ev.score)

        if iteration < self.max_retries:
            print(f"  [iteration {iteration}] Retrying with feedback: {ev.feedback}")
            return GeneratePromptEvent(feedback=ev.feedback)

        print(f"  [iteration {iteration}] Max retries reached — escalating to human.")
        return HumanFeedbackRequired(prompt=ev.prompt, history=history)

    # -- Step 3: Approval — terminal step -----------------------------------

    @step
    async def handle_approval(
        self, ctx: Context[RefinementState], ev: ApprovedEvent
    ) -> StopEvent:
        state = await ctx.store.get_state()
        print(f"  Approved after {state.iteration} iteration(s) with score {ev.score}.")
        return StopEvent(
            result={
                "status": "approved",
                "prompt": ev.prompt,
                "score": ev.score,
                "iterations": state.iteration,
            }
        )

## 5. Run: Auto-Approve Path

With `max_retries=5` and `threshold=0.8`, the mock evaluator will approve
once the prompt accumulates enough quality keywords (4 out of 5 criteria = 0.8).
Since the generator adds at most 2 improvements per iteration, this takes ~3 iterations.

In [6]:
evaluator = MockEvaluator(dataset=DATASET)
wf = PromptRefinementWorkflow(
    evaluator=evaluator, max_retries=5, threshold=0.8, timeout=30
)

print("=== Auto-Approve Run ===")
result = await wf.run()
print(f"\nResult: {result}")

=== Auto-Approve Run ===
  [iteration 1] Generated prompt: 'Answer the question.'
  [iteration 1] Score: 0.0 (threshold: 0.8)
  [iteration 1] Retrying with feedback: Missing qualities: specific, structured, role, examples, constraints. Try adding language that is specific, structured, role, examples, constraints.
  [iteration 2] Generated prompt: 'Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer.'
  [iteration 2] Score: 0.4 (threshold: 0.8)
  [iteration 2] Retrying with feedback: Missing qualities: role, examples, constraints. Try adding language that is role, examples, constraints.
  [iteration 3] Generated prompt: 'Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer. You are a helpful and knowledgeable assistant. For instance, if asked about math, show your working (e.g. 2+2=4).'
  [iteration 3] Score: 0.8 (threshold: 0.8)
  Approv

## 6. Run: HITL Escalation Path

Here we set `max_retries=2` so the workflow can only loop twice before
escalating. With only 2 iterations adding 2 improvements each, the prompt
reaches at most 4/5 criteria (score 0.8) which is below `threshold=1.0`.
This forces escalation to a human.

We consume events from the handler, simulate a human providing targeted
feedback, and watch the loop resume.

In [7]:
evaluator = MockEvaluator(dataset=DATASET)
wf = PromptRefinementWorkflow(
    evaluator=evaluator, max_retries=2, threshold=1.0, timeout=30
)

print("=== HITL Escalation Run ===")
handler = wf.run()

async for ev in handler.stream_events():
    if isinstance(ev, HumanFeedbackRequired):
        print(f"\n  HITL requested! Current prompt: {ev.prompt!r}")
        print(f"  History ({len(ev.history)} attempts):")
        for entry in ev.history:
            print(f"    score={entry['score']} | {entry['prompt'][:80]}")

        # Simulate a human providing targeted feedback
        human_feedback = "specific, structured, role, examples, constraints"
        print(f"\n  Sending human feedback: {human_feedback!r}")
        handler.ctx.send_event(HumanFeedbackResponse(response=human_feedback))

result = await handler
print(f"\nResult: {result}")

=== HITL Escalation Run ===
  [iteration 1] Generated prompt: 'Answer the question.'
  [iteration 1] Score: 0.0 (threshold: 1.0)
  [iteration 1] Retrying with feedback: Missing qualities: specific, structured, role, examples, constraints. Try adding language that is specific, structured, role, examples, constraints.
  [iteration 2] Generated prompt: 'Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer.'
  [iteration 2] Score: 0.4 (threshold: 1.0)
  [iteration 2] Max retries reached — escalating to human.

  HITL requested! Current prompt: 'Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer.'
  History (2 attempts):
    score=0.0 | Answer the question.
    score=0.4 | Answer the question. Be precise and concise in your answer. First, understand th

  Sending human feedback: 'specific, structured, role, examples, constraints'
  Human fee

## 7. Inspect State

After a run, we can read the workflow state to see the full refinement history
— including the attempts before and after human intervention.

In [8]:
state = await handler.ctx.store.get_state()

print(f"Total iterations: {state.iteration}")
print(f"Final prompt: {state.current_prompt!r}")
print(f"\nFull history ({len(state.history)} entries):")
for i, entry in enumerate(state.history, 1):
    print(f"  {i}. score={entry['score']:.2f} | {entry['prompt']}")

Total iterations: 2
Final prompt: 'Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer. You are a helpful and knowledgeable assistant. For instance, if asked about math, show your working (e.g. 2+2=4). You must answer accurately. Do not guess. Always verify your reasoning.'

Full history (4 entries):
  1. score=0.00 | Answer the question.
  2. score=0.40 | Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer.
  3. score=0.80 | Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer. You are a helpful and knowledgeable assistant. For instance, if asked about math, show your working (e.g. 2+2=4).
  4. score=1.00 | Answer the question. Be precise and concise in your answer. First, understand the question. Then, provide a step-by-step answer. You are a helpful and knowle

## Summary

This example demonstrated four workflow patterns working together:

| Pattern | How it was used |
|---------|----------------|
| **Looping** | `evaluate_and_decide` returns `GeneratePromptEvent` to re-enter `generate` |
| **Branching** | `evaluate_and_decide` chooses between `ApprovedEvent`, `GeneratePromptEvent`, or `HumanFeedbackRequired` |
| **State management** | `RefinementState` tracks iteration count, prompt history, and current prompt via `ctx.store` |
| **Human-in-the-loop** | At max retries, `evaluate_and_decide` emits `HumanFeedbackRequired`; the caller responds with `HumanFeedbackResponse`, which re-enters `generate` directly |

### Next steps

- Replace `MockEvaluator` with a real LLM-based judge (e.g., OpenAI)
- Replace `generate_prompt()` with an LLM call that takes feedback as input
- Add parallel evaluation of multiple prompt candidates using `ctx.send_event()` + `ctx.collect_events()`
- Persist state across restarts using `ctx.to_dict()` / `Context.from_dict()`

See also:
- [Feature Walkthrough](feature_walkthrough.ipynb) for individual pattern walkthroughs
- [Durable Workflows](durable_workflows.ipynb) for state persistence
- [Document Processing](document_processing.ipynb) for a real-world HITL example with LlamaCloud